# IELTS Document Extraction Regression

Temporary Colab notebook for extraction quality work. It does not start Ollama, FastAPI, the frontend, or a Cloudflare tunnel.

Pipeline under test: native extraction -> layout -> OCR -> reconciliation -> structure parsing -> visual parsing -> chunking.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/phamtu2x5/ielts-chatbot.git"
BRANCH = "main"
REPO_DIR = Path("/content/ielts-chatbot")
OUTPUT_DIR = Path("/content/extraction_regression_output")
PREFLIGHT_DIR = Path("/content/extraction_regression_preflight")
SKIP_MODEL_WARMUP = False

print("Repository:", REPO_URL)
print("Branch:", BRANCH)
print("Regression output:", OUTPUT_DIR)

## 1. Sync repository and Git LFS fixtures

In [ ]:
import shutil
import subprocess

def run(command, *, cwd=None, check=True):
    print("$", " ".join(str(part) for part in command))
    return subprocess.run([str(part) for part in command], cwd=cwd, check=check)

if shutil.which("git-lfs") is None:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])

if (REPO_DIR / ".git").exists():
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

run(["git", "lfs", "install", "--local"], cwd=REPO_DIR)
run(["git", "lfs", "pull"], cwd=REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Checked out commit:", commit)

## 2. Verify the Colab GPU runtime

In [ ]:
import subprocess
import sys

run(["nvidia-smi"])
try:
    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM MiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**2, 2))
except ImportError:
    print("Torch will be checked again after installing requirements.")

## 3. Install extraction dependencies only

In [ ]:
import os
import sys

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade-strategy",
    "only-if-needed",
    "-r",
    REPO_DIR / "backend" / "requirements.txt",
])

os.environ.update({
    "OCR_ENGINE": "rapidocr",
    "OCR_RUNTIME": "torch",
    "OCR_DEVICE": "cuda:0",
    "OCR_VERSION": "PP-OCRv6",
    "OCR_MODEL_SIZE": "medium",
    "LAYOUT_ENABLE": "true",
    "LAYOUT_ENGINE": "doclayout_yolo",
    "LAYOUT_DEVICE": "cuda:0",
    "WARMUP_OCR": "true",
    "WARMUP_LAYOUT": "true",
})

import torch
assert torch.cuda.is_available(), "A CUDA Colab runtime is required for this regression notebook."
print("Extraction dependencies are ready on", torch.cuda.get_device_name(0))

## 4. Preflight fixtures before loading OCR/layout models

In [ ]:
import json
import shutil

shutil.rmtree(PREFLIGHT_DIR, ignore_errors=True)
run([
    sys.executable,
    REPO_DIR / "backend" / "tools" / "extraction_regression.py",
    "--preflight-only",
    "--no-archive",
    "--output-dir",
    PREFLIGHT_DIR,
    "--overwrite",
])
preflight = json.loads((PREFLIGHT_DIR / "preflight.json").read_text())
assert preflight["ok"], json.dumps(preflight, indent=2)
print("Verified fixtures:", len(preflight["fixtures"]))

## 5. Run extraction regression directly

A non-zero exit code means at least one quality check failed. The debug bundle is still created.

In [ ]:
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
command = [
    sys.executable,
    REPO_DIR / "backend" / "tools" / "extraction_regression.py",
    "--output-dir",
    OUTPUT_DIR,
    "--overwrite",
]
if SKIP_MODEL_WARMUP:
    command.append("--skip-warmup")
result = run(command, check=False)
print("Regression exit code:", result.returncode)
print("The summary and ZIP are available even when checks fail.")

## 6. Inspect summary

In [ ]:
summary_path = OUTPUT_DIR / "regression_summary.json"
summary = json.loads(summary_path.read_text())
print("Overall status:", summary.get("status"))
print("Counts:", json.dumps(summary.get("counts", {}), indent=2))
for document in summary.get("documents", []):
    print(f"- {document['status']:11s} {document['duration_seconds']:>7.2f}s  {document['filename']}")
    for check in document.get("checks", []):
        if check.get("status") != "passed":
            print("   ", check.get("status"), check.get("name"), check.get("details", ""))

## 7. Download the debug bundle

In [ ]:
from google.colab import files

archives = sorted(
    OUTPUT_DIR.parent.glob("extraction-regression-debug-*.zip"),
    key=lambda path: path.stat().st_mtime,
)
assert archives, "Regression archive was not created."
archive = archives[-1]
print("Downloading:", archive)
files.download(str(archive))